# 03 - Engenharia de features (Fase 3 - seleção de modelos)

**Regras que seguimos:**
1. **Uma feature herda a elegibilidade das colunas de onde vem**. Ex.: tudo o que usa `installment` (T2, *pricing*) só entra na classificação; tudo o que usa `issue_d` (T4) também.
2. **As features ficam dentro do pipeline**: o `FeatureBuilder` (`src/lending/feature_engineering.py`) é um transformer do scikit-learn e vem antes do `build_preprocessor`. Os cálculos são linha a linha e não aprendem nada com os dados, mas assim são aplicados sempre da mesma forma no treino, na validação e no teste.
3. **Divisões por zero e datas impossíveis dão NaN**. O pipeline trata esses NaN como os das outras colunas: mediana do treino de cada *fold* + indicador de "em falta".
4. Só dados de desenvolvimento. O teste final (2015) não é aberto.

**Divisão no grupo:** 5 features de cada elemento.

**Antes de correr:** é preciso ter corrido `01_preprocessamento.ipynb` (gera `regressao_dev.pkl` e `classificacao_treino.pkl`).

In [ ]:
# Setup
import sys
from pathlib import Path

# Tenta importar o lending
try:
    import lending
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "src" / "lending").is_dir():
            sys.path.insert(0, str(parent / "src"))
            break
    import lending

import dataclasses

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from lending import config
from lending.feature_engineering import (FEATURES, FeatureBuilder, add_engineered, catalog,
                                         check_leakage, input_columns, specs_for)
from lending.features import TASKS, TaskFeatures
from lending.preprocessing import ExpandingWindowSplit, build_preprocessor
from lending.storage import PickleStore

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 1. Fichas das features

Cada feature é um `FeatureSpec` que contem: colunas de origem, fórmula, motivo, tratamento dos valores em falta e momento a partir do qual existe (T0 pedido, T2 *pricing*, T4 concessão; ver `docs/01_formulacao_problema/momentos_previsao.md`).

In [ ]:
catalog()

## 2. Elegibilidade por tarefa e verificação contra *leakage*

| Tarefa | Momento de previsão | Que features entram |
|---|---|---|
| Regressão | T0 (antes do *pricing*) | só as que usam colunas de T0 |
| Classificação | T4 (concessão) | todas as que usam colunas até T4 |
| Clustering | perfil em T0 | só as de T0 |

O `check_leakage` usa as listas de colunas proibidas do enunciado (`features.py`). Se alguma feature de uma tarefa usar uma coluna proibida nessa tarefa dá erro. A última célula mostra o erro que aparece se alguém tentar usar `installment_to_income` na regressão.

In [ ]:
for task in TASKS:
    check_leakage(task)
    print(f"{task:15s} {len(specs_for(task))} features: {[s.name for s in specs_for(task)]}")

In [ ]:
# Demonstração: a mesma feature, marcada (por engano) como elegível na regressão, é recusada
wrong = [dataclasses.replace(FEATURES[1], tasks=("regression", "classification"))]
try:
    check_leakage("regression", wrong)
except ValueError as error:
    print("Recusada:", error)

## 3. Criar as features nos dados de desenvolvimento

`input_columns` junta às colunas de entrada de cada tarefa as colunas de origem que só entram transformadas (na classificação, `issue_d` e `earliest_cr_line`). Depois do `FeatureBuilder` o `build_preprocessor` descarta-as.

In [ ]:
regression = PickleStore.load(config.REG_DEV_PKL, expected_type=pd.DataFrame)
classification = PickleStore.load(config.CLS_TRAIN_PKL, expected_type=pd.DataFrame)
data = {"regression": regression, "classification": classification}

base = {task: TaskFeatures(task) for task in data}
built = {}
for task in data:
    builder = FeatureBuilder(task).fit(data[task][input_columns(base[task])])
    built[task] = builder.transform(data[task][input_columns(base[task])])
    print(f"{task}: {len(data[task]):,} linhas; features novas: {builder.names_}")

## 4. Verificações

- percentagem de valores em falta;
- nenhum infinito;
- distribuição de cada feature, para confirmar que os valores fazem sentido;
- regras de coerência: o DTI com o novo empréstimo nunca é menor do que o DTI; a antiguidade do crédito nunca é negativa; o ano está dentro do período de treino.

In [ ]:
for task, X in built.items():
    names = [s.name for s in specs_for(task)]
    table = X[names].describe(percentiles=[0.01, 0.5, 0.99]).T
    table.insert(1, "falta_%", X[names].isna().mean().mul(100))
    table.insert(2, "infinitos", np.isinf(X[names]).sum())
    print(task)
    display(table.round(2))

In [ ]:
X = built["classification"]
both = X[["dti", "dti_with_loan"]].dropna()
assert (both["dti_with_loan"] >= both["dti"]).all()
assert (X["credit_history_years"].dropna() >= 0).all()
assert X["issue_year"].between(2007, 2011).all()
assert not np.isinf(X[[s.name for s in specs_for("classification")]]).any().any()
print("Regras de coerência verificadas.")

## 5. Relação com o alvo (só treino, descritivo)

Para confirmar que as features têm significado, compara-se cada uma com o alvo **nos dados de treino**. É só descrição, pois a escolha final das features é feita dentro da validação cruzada.

In [ ]:
# Regressão: correlação de Spearman com int_rate
X = built["regression"]
pd.Series({name: X[name].corr(regression["int_rate"], method="spearman")   # mesmo índice
           for name in [s.name for s in specs_for("regression")]}, name="rho_int_rate").round(3)

In [ ]:
# Classificação: correlação com o default e taxa de default (%) por quintil de cada feature
X = built["classification"]
is_default = (classification[config.CLASS_COL] == "Default").astype(int)   # mesmo índice que X
names = [s.name for s in specs_for("classification")]
display(pd.Series({n: X[n].corr(is_default, method="spearman") for n in names},
                  name="rho_default").round(3))

quintiles = {}
for n in names:
    rates = is_default.groupby(pd.qcut(X[n], q=5, duplicates="drop"), observed=True).mean()
    quintiles[n] = pd.Series(rates.mul(100).round(1).to_numpy(),
                             index=[f"Q{i}" for i in range(1, len(rates) + 1)])
pd.DataFrame(quintiles).T.rename_axis("% default por quintil")

## 6. As features no pipeline (*fit* só no treino de cada *fold*)

O pipeline completo é `FeatureBuilder` -> `build_preprocessor`. Em cada *fold* da validação temporal é ajustado de novo, só com o treino desse *fold*, e aplicado à validação. As features novas passam pelos mesmos passos das outras numéricas, *clipping* aos percentis do treino, imputação pela mediana + indicador e padronização.

In [ ]:
splitters = {"regression": ExpandingWindowSplit("Q", min_train_periods=2),
             "classification": ExpandingWindowSplit("Y", min_train_periods=2)}
rows = []
for task, splitter in splitters.items():
    cols, extended = input_columns(base[task]), add_engineered(base[task])
    for fold, (train_idx, valid_idx) in enumerate(splitter.split(data[task]["issue_d"]), start=1):
        pipe = Pipeline([("features", FeatureBuilder(task)), ("pre", build_preprocessor(extended))])
        X_train = pipe.fit_transform(data[task].iloc[train_idx][cols])
        X_valid = pipe.transform(data[task].iloc[valid_idx][cols])
        assert X_train.notna().all().all() and X_valid.notna().all().all()
        new = [n for n in pipe["features"].names_ if n in X_train.columns]
        rows.append({"tarefa": task, "fold": fold, "linhas treino": len(train_idx),
                     "linhas validação": len(valid_idx), "colunas de saída": X_train.shape[1],
                     "features novas na saída": len(new)})
pd.DataFrame(rows).set_index(["tarefa", "fold"])

## 7. Resumo

| O quê | Onde |
|---|---|
| Fichas das features | `catalog()`, `docs/03_features/features.md`, secção 3 do relatório |
| Cálculo | `FeatureBuilder` (`src/lending/feature_engineering.py`) |
| Verificação contra *leakage* | `check_leakage` (listas do enunciado em `features.py`) |
| Entrada no pipeline | `add_engineered` + `input_columns` + `build_preprocessor` |

**Como se usa na Parte 2:**

```python
base = TaskFeatures("classification")
pipe = Pipeline([("features", FeatureBuilder("classification")),
                 ("pre", build_preprocessor(add_engineered(base))),
                 ("model", modelo)])
pipe.fit(treino[input_columns(base)], y_treino)
```